## PCA plotting of MCE features

### Imports and preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tifffile as tiff
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree
from tqdm import tqdm

In [ ]:
#Data normalization function
def normalize_dataframe(df, columns_to_normalize):
    # Create a copy of the dataframe
    df_normalized = df.copy()

    # Replace missing values with 0 in the selected columns
    #df_normalized[columns_to_normalize] = df_normalized[columns_to_normalize].fillna(0)

    # Normalize selected columns
    scaler = StandardScaler()
    scaled_values = scaler.fit_transform(df_normalized[columns_to_normalize].values)
    df_normalized[columns_to_normalize] = scaled_values

    return df_normalized

In [ ]:
full_df = pd.read_csv('D:/Mari_Sixth_Dataset_Analysis/full_df_for_classification.csv')

In [ ]:
full_df_seconddataset = pd.read_csv('D:/Mari_Second_Dataset_Analysis/full_df_for_classification.csv')

In [ ]:
columns_to_normalize = [
#'Local_Cell_Density', 
 'Speed',
 'Acceleration',
 'Motion_Angle_Z',
 'Motion_Angle_Y',
 'Motion_Angle_X',
 'DISPLACEMENT_Z',
 'DISPLACEMENT_Z_ABS',
 'DISPLACEMENT_Z_SUMMED',
 'Displacement',
 'Edge_xy_angle',
 'Directional_Change',
 'Directional_Change_Abs',
 'Displacement_summed',
 'Directional_Change_Summed',
 'nuc_Radius',
 'nuc_Surface_Area',
 'nuc_Eccentricity_Comp_First',
 'nuc_Eccentricity_Comp_Second',
 'nuc_Eccentricity_Comp_Third',
 'nuc_Cell_Axis_Z',
 'nuc_Cell_Axis_Y',
 'nuc_Cell_Axis_X',
 'mem_nuc_offset',
 'mem_Radius',
 'mem_Surface_Area',
 'mem_Eccentricity_Comp_First',
 'mem_Eccentricity_Comp_Second',
 'mem_Eccentricity_Comp_Third',
 'mem_Cell_Axis_Z',
 'mem_Cell_Axis_Y',
 'mem_Cell_Axis_X',
 'mem_2d_area',
 'mem_2d_perimeter',
 'mem_2d_eccentricity',
 'mem_2d_solidity',
 'mem_2d_extent',
 'mem_2d_axis_major_length',
 'mem_2d_axis_minor_length',
 'mem_2d_feret_diameter_max',
 'Distance_Cell_mask',
 'POSITION_Z_norm',
 'Radial_Angle_Z',
 'Radial_Angle_Y',
 'Radial_Angle_X',
 #'MSD',
# 'mem_Radial_Angle_Z',
# 'centroid_to_cell_angle',
 'angle_difference',
 'angle_difference_summed']

In [ ]:
# The features are z normalized for training
full_df_normalized = normalize_dataframe(full_df, columns_to_normalize)

In [ ]:
# The features are z normalized for training
full_df_normalized_seconddataset = normalize_dataframe(full_df_seconddataset, columns_to_normalize)

In [ ]:
full_df2 = full_df_normalized[full_df_normalized['cell_type'] != 'unknown'].dropna(subset=columns_to_normalize)

In [ ]:
full_df2_seconddataset = full_df_normalized_seconddataset[full_df_normalized_seconddataset['cell_type'] != 'unknown'].dropna(subset=columns_to_normalize)

In [ ]:
# Count unique t_hours per Track ID and cell_type
track_t_hours_counts = full_df2[full_df2['annotation']  == 'manual'].groupby(['cell_type', 'Spot track ID relabelled'])['t_hours'].nunique().reset_index(name='unique_t_hours')

# Find the Track ID with the highest unique t_hours count per cell_type
result = track_t_hours_counts.loc[track_t_hours_counts.groupby('cell_type')['unique_t_hours'].idxmax(), ['cell_type', 'Spot track ID relabelled', 'unique_t_hours']]


In [ ]:
# hand-selecting lineages so that daughter cells match mother cells, but also that the lineage is as long as possible
selected_df = full_df2[full_df2['Track ID_y'].isin([35700, 35712, 35725, 239900, 239911, 21800, 21811, 21823, 4800, 4812, 196000, 196011, 196024, 196037])]

In [ ]:
selected_df['cell_type'].value_counts()

In [ ]:
feature_categories = {'membrane_shape' : ['mem_Radius',
 'mem_Eccentricity_Comp_First',
 'mem_Eccentricity_Comp_Second',
 'mem_Eccentricity_Comp_Third',
 'mem_Surface_Area',
 'mem_Cell_Axis_Z',
 'mem_2d_area',
 'mem_2d_eccentricity',
 'mem_2d_solidity',
 'mem_2d_extent'],

'nucleus_shape' : ['nuc_Radius',
 'nuc_Eccentricity_Comp_First',
 'nuc_Eccentricity_Comp_Second',
 'nuc_Eccentricity_Comp_Third',
 'nuc_Cell_Axis_Z',
 'mem_nuc_offset'],

 'position' : [
 'POSITION_Z_norm',
 'Distance_Cell_mask',
 #'Radial_Angle_Z',
 # 'Radial_Angle_Y',
 #'Radial_Angle_X',
 ],

'movement' : ['Speed',
 'Motion_Angle_Z',
 'Motion_Angle_Y',
 'Motion_Angle_X',
 'Acceleration',
 'DISPLACEMENT_Z_ABS',
'DISPLACEMENT_Z_SUMMED',
'Displacement', 
'Directional_Change_Abs',
'Displacement_summed',
'Directional_Change_Summed',
 'angle_difference',
 'angle_difference_summed']
}

In [ ]:
# Flatten the list of features preserving order
ordered_features = [f for features in feature_categories.values() for f in features]

In [ ]:
full_df_normalized_cropped = full_df_normalized.dropna(subset=columns_to_normalize)
full_df_normalized_cropped2 = full_df_normalized_cropped[ordered_features]

In [ ]:
full_df_normalized_cropped_seconddataset = full_df_normalized_seconddataset.dropna(subset=columns_to_normalize)
full_df_normalized_cropped2_seconddataset = full_df_normalized_cropped_seconddataset[ordered_features]

In [ ]:
random_choices = np.random.choice(full_df_normalized_cropped['Spot track ID relabelled'].value_counts()[full_df_normalized_cropped['Spot track ID relabelled'].value_counts() > 200].index.to_list(), size=3, replace=False)

random_choices 


In [ ]:
random_choices_tracklets = [31700, 31711, 31724, 196900, 344900]

In [ ]:
import matplotlib.patches as mpatches

In [ ]:
import matplotlib as mpl

plt.rcdefaults()

# Set global font style and sizes      
mpl.rcParams['font.size'] = 16                 # Base font size
mpl.rcParams['axes.titlesize'] = 16            # Title font size
mpl.rcParams['axes.labelsize'] = 16            # Axis label font size
mpl.rcParams['xtick.labelsize'] = 16           # X-tick font size
mpl.rcParams['ytick.labelsize'] = 16           # Y-tick font size
mpl.rcParams['legend.fontsize'] = 14            # Legend font size
mpl.rcParams['figure.titlesize'] = 16          # Figure title font size

#reset mpl.rcParams font to default
mpl.rcParams['font.family'] = 'DejaVu Sans'

In [ ]:
from tqdm import tqdm

### Fig 3F and Appendix Figure S3

In [ ]:
# Count original number of rows
original_count = len(full_df_normalized)

# Create a mask where all selected feature values are ≤ 10
mask = (full_df_normalized[columns_to_normalize] <= 10).all(axis=1)

# Filter the DataFrame
full_df_normalized_filtered = full_df_normalized[mask]

# Count how many rows were removed
removed_count = original_count - len(full_df_normalized_filtered)

print(f"Removed {removed_count} rows where any selected feature exceeded 10.")


In [ ]:
# Loop through selected features
for feature in ordered_features:
    plt.figure(figsize=(5, 3))
    sns.lineplot(
        data=full_df_normalized_filtered,
        x='t_hours',
        y=feature,
        errorbar='sd'  # use 'ci' for confidence interval, or 'sd' for standard deviation
    )
    plt.xlabel('Time (hours)')
    plt.ylabel('Standardized value')
    plt.title(f'{feature}')
    #plt.legend()
    plt.tight_layout()
    #plt.savefig(f'mean_feature_plots/{feature}.pdf')
    plt.show()

In [ ]:
ordered_names = {'mem_Radius' : 'Membrane radius',
 'mem_Eccentricity_Comp_First' : 'Membrane major axis length', 
 'mem_Eccentricity_Comp_Second' : 'Membrane intermediate axis length',
 'mem_Eccentricity_Comp_Third' : 'Membrane minor axis length',
 'mem_Surface_Area' : 'Membrane surface area',
 'mem_Cell_Axis_Z' : 'Membrane cell axis Z',   
 'mem_2d_area' : 'Membrane center slice area',
 'mem_2d_eccentricity' : 'Membrane center slice eccentricity',
 'mem_2d_solidity' : 'Membrane center slice solidity',
 'mem_2d_extent' : 'Membrane center slice extent',
 'nuc_Radius' : 'Nucleus radius',
 'nuc_Eccentricity_Comp_First' : 'Nucleus major axis length',
 'nuc_Eccentricity_Comp_Second' : 'Nucleus intermediate axis length',   
 'nuc_Eccentricity_Comp_Third' : 'Nucleus minor axis length',
 'nuc_Cell_Axis_Z' : 'Nucleus cell axis Z',
 'mem_nuc_offset' : 'Membrane-nucleus offset',
 'POSITION_Z_norm' : 'Normalized Z position',
 'Distance_Cell_mask' : ' Distance to tissue boundary',
 'Radial_Angle_Z' : 'Radial angle Z',
 'Radial_Angle_Y' : 'Radial angle Y',
 'Radial_Angle_X' : 'Radial angle X',
 'Speed' : 'Speed',
 'Motion_Angle_Z' : 'Motion angle Z',
 'Motion_Angle_Y' : 'Motion angle Y',
 'Motion_Angle_X' : 'Motion angle X',
 'Acceleration' : 'Acceleration',
 'DISPLACEMENT_Z_ABS' : 'Z displacement',
 'DISPLACEMENT_Z_SUMMED' : 'Summed Z displacement',
 'Displacement' : 'Displacement',
 'Directional_Change_Abs' : 'Directional change',
 'Displacement_summed': 'Summed displacement',
 'Directional_Change_Summed' : 'Summed directional change',
 'angle_difference' : 'Radial divergence',
 'angle_difference_summed' : 'Summed radial divergence'}

In [ ]:
import math

mpl.rcParams['font.family'] = 'Arial'

# Set A4 figure size in inches (width x height)
a4_width, a4_height = 8.27, 11.69

# Set Arial globally
#mpl.rcParams['font.family'] = 'Arial'

# Number of subplots
n_features = len(ordered_features)
n_cols = 4
n_rows = math.ceil(n_features / n_cols)

# Adjust subplot spacing based on content
fig, axes = plt.subplots(n_rows, n_cols, figsize=(a4_width, a4_height))#, constrained_layout=True)
axes = axes.flatten()

for i, feature in enumerate(ordered_features):
    ax = axes[i]
    sns.lineplot(
        data=full_df_normalized_filtered,
        x='t_hours',
        y=feature,
        errorbar='sd',
        ax=ax
    )
    ax.set_title(ordered_names[feature], fontsize=8)
    ax.set_xlabel('Time (hours)', fontsize=7)
    ax.set_ylabel('Standardized feature', fontsize=7)
    ax.tick_params(labelsize=5)

# Hide any unused axes
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Standardized features, timewise mean ± sd", fontsize=10)

# Tighten layout manually
plt.subplots_adjust(
    left=0.05,   # Reduce left margin
    right=0.98,  # Reduce right margin
    top=0.94,    # Leave some room for title
    bottom=0.06, # Leave some room for x-labels
    hspace=0.8,  # Reduce vertical spacing
    wspace=0.3   # Reduce horizontal spacing
)

#plt.savefig((f'mean_feature_plots_subplots.png'), bbox_inches='tight', dpi=300)
plt.show()


### Appendix Figure S5

In [ ]:
manual_df = full_df_normalized[full_df_normalized['cell_type'] != 'unknown']

In [ ]:
# Group by both cell_type and t_hours, and count unique Track ID_y values
group_counts = manual_df.groupby(['cell_type', 't_hours'])['Track ID_y'].nunique().reset_index()

# Keep only groups with at least 3 unique Track IDs
valid_groups = group_counts[group_counts['Track ID_y'] >= 3]

# Merge to filter the original dataframe
manual_df_filtered = manual_df.merge(valid_groups[['cell_type', 't_hours']], on=['cell_type', 't_hours'], how='inner')

# Create a mask where all selected feature values are ≤ 10
mask = (manual_df_filtered[ordered_features] <= 10).all(axis=1)

# Filter the DataFrame
manual_df_filtered2 = manual_df_filtered[mask]


In [ ]:
# set global font to default
mpl.rcParams['font.family'] = 'DejaVu Sans'

In [ ]:
# Ensure 't_hours' is sorted correctly
manual_df_filtered2 = manual_df_filtered2.sort_values(['cell_type', 't_hours'])

# Define a custom color dictionary for cell types
custom_colors = {
    'basal': 'royalblue',  
    'goblet': 'darkorange',  
    'ic': 'magenta',  
    'mcc': 'green',
    'ssc' : 'gold'   
}

for feature in ordered_features:
    # Create the line plot
    plt.figure(figsize=(5, 3))
    sns.lineplot(
        data=manual_df_filtered2,
        x='t_hours', 
        y=feature, 
        hue='cell_type',  # Different colors for each cell type
        palette=custom_colors,
        estimator='mean',  # Averaging Z_norm for each (cell_type, t_hours)
        errorbar='sd',
        alpha=0.8
    )

    # Customize labels and title
    plt.xlabel('Time (hours)')
    plt.ylabel('Standardized feature')
    plt.title(f'{feature}')
    plt.legend().remove()
    plt.grid(False)
    #plt.savefig(f'celltype_mean_feature_plots/celltype_mean_{feature}.pdf')
    plt.show()

In [ ]:
# Ensure data is sorted
manual_df_filtered2 = manual_df_filtered2.sort_values(['cell_type', 't_hours'])

# A4 size in inches
a4_width, a4_height = 8.27, 11.69

# Grid layout
n_features = len(ordered_features)
n_cols = 4  # fewer columns for better space if legend is added
n_rows = math.ceil(n_features / n_cols)

# Color map for cell types
custom_colors = {
    'basal': 'royalblue',  
    'goblet': 'orangered',  
    'ic': 'magenta',  
    'mcc': 'green',
    'ssc': 'goldenrod'
}

# Create figure and axes
fig, axes = plt.subplots(n_rows, n_cols, figsize=(a4_width, a4_height))
axes = axes.flatten()

for i, feature in enumerate(ordered_features):
    ax = axes[i]
    sns.lineplot(
        data=manual_df_filtered2,
        x='t_hours',
        y=feature,
        hue='cell_type',
        palette=custom_colors,
        estimator='mean',
        errorbar='sd',
        linewidth=0.6,
        alpha=0.7,
        ax=ax
    )
    ax.set_title(ordered_names[feature], fontsize=8)
    ax.set_xlabel('Time (hours)', fontsize=7)
    ax.set_ylabel('Standardized feature', fontsize=7)
    ax.tick_params(labelsize=5)
    ax.legend().remove()
    ax.grid(False)

# Hide extra axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

# Add a shared legend outside the plot
handles, labels = axes[0].get_legend_handles_labels()

#make legend with custom placement
fig.legend(
    handles, 
    labels, 
    loc='upper center', 
    ncol=len(custom_colors), 
    fontsize=6,
    bbox_to_anchor=(0.5, 0.96), # X=center, Y=slightly below suptitle
    frameon=False               # Optional: removes the box for a cleaner look
)

# Adjust layout
plt.subplots_adjust(
    left=0.05,
    right=0.98,
    top=0.92,
    bottom=0.06,
    hspace=0.8,
    wspace=0.3
)

fig.suptitle("Timewise feature trends by cell type", fontsize=10)
#plt.savefig((f'mean_feature_plots_subplots_celltypes.png'), bbox_inches='tight', dpi=300)
plt.show()

### Figure 3D, Figure EV1B and Figure EV2B (membrane features)

In [ ]:
#set global font to default
mpl.rcParams['font.family'] = 'DejaVu Sans'

# Fit and transform PCA
pca = PCA(n_components=2)
mem_shape_pca_result = pca.fit_transform(full_df_normalized_cropped2[[feature for feature in ordered_features if feature in feature_categories['membrane_shape']]])  # Drop categorical column

# Convert PCA results to DataFrame
pca_df = pd.DataFrame(mem_shape_pca_result, columns=['PC1', 'PC2'])

pca_df['t_hours'] = full_df_normalized_cropped['t_hours'].values
pca_df['Spot track ID relabelled'] = full_df_normalized_cropped['Spot track ID relabelled'].values
pca_df['cell_type'] = full_df_normalized_cropped['cell_type'].values

pca_df_selected_annotated = pca_df[(pca_df['cell_type'] != 'unknown')]

pca_df_selected_annotated = pca_df_selected_annotated.sort_values(['cell_type', 't_hours']).reset_index(drop=True)

pca_df_selected = pca_df[(pca_df['Spot track ID relabelled'].isin(random_choices))]

pca_df_selected = pca_df_selected.sort_values(['Spot track ID relabelled', 't_hours']).reset_index(drop=True)

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df['PC1'], pca_df['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df['t_hours'],  # Color by time
    cmap='viridis',
    rasterized = True
)

plt.colorbar(label='Time (hours)')  # Add color legend for time
plt.title('Membrane shape PCA by time')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-10,10))
plt.ylim((-10,8))
#plt.savefig('plots_15_03/mem_shape_PCA.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

plt.figure(figsize=(4, 4))

for track_id, group in pca_df_selected.groupby('Spot track ID relabelled'):  
    plt.plot(group["PC1"], group["PC2"], label=f'Track {track_id}', linewidth=2, alpha=0.7)

plt.title('Tracks in membrane shape PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-10,10))
plt.ylim((-10,8))
plt.legend()
#plt.savefig('plots_15_03/mem_shape_PCA_tracks.pdf')
plt.show()

# Define color mapping for cell types
cell_type_colors = {'basal': 'royalblue', 'goblet': 'orange', 'ic': 'magenta', 'mcc': 'green', 'ssc': 'gold'}

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df_selected_annotated['PC1'], pca_df_selected_annotated['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df_selected_annotated['cell_type'].map(cell_type_colors),  # Color by cell type
    rasterized = True
)

# Create legend handles
legend_patches = [mpatches.Patch(color=color, label=ctype) for ctype, color in cell_type_colors.items()]
plt.legend(handles=legend_patches, title="Cell type", frameon=True)

plt.title('Membrane shape PCA by cell type')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-10,10))
plt.ylim((-10,8))
#plt.savefig('plots_15_03/mem_shape_PCA_celltypes.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()


### To figure EV2A (plotted later)

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier


In [ ]:
def knn_error(X, y, k=5):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X, y.ravel())
    return 1 - knn.score(X, y.ravel())

def knn_cv_accuracy(X, y, k=5, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    accuracies = []
    balanced_accuracies = []

    for train_index, test_index in skf.split(X, y.ravel()):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        clf = KNeighborsClassifier(n_neighbors=k)
        clf.fit(X_train, y_train.ravel())
        y_pred = clf.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        accuracies.append(acc)
        balanced_acc = balanced_accuracy_score(y_test, y_pred)
        balanced_accuracies.append(balanced_acc)
    
    return accuracies, balanced_accuracies

from sklearn.dummy import DummyClassifier

def kmeans_ari(X, y, n_clusters=None):
    if n_clusters is None:
        n_clusters = len(np.unique(y))
    km = KMeans(n_clusters=n_clusters, random_state=0)
    labels = km.fit_predict(X)
    return adjusted_rand_score(y.ravel(), labels)

def kmeans_nmi(X, y, n_clusters=None):
    if n_clusters is None:
        n_clusters = len(np.unique(y))
    km = KMeans(n_clusters=n_clusters, random_state=0)
    labels = km.fit_predict(X)
    return normalized_mutual_info_score(y.ravel(), labels)

def calc_asw(X, y):
    return silhouette_score(X, y.ravel())

In [ ]:
from sklearn.preprocessing import StandardScaler
#from metrics import knn_error, kmeans_nmi, kmeans_ari, calc_asw
from tqdm import tqdm

metrics = []

for t in tqdm(pca_df_selected_annotated['t_hours'].unique()):
    cur_df = pca_df_selected_annotated[pca_df_selected_annotated['t_hours'] == t]
    
    X = StandardScaler().fit_transform(cur_df[['PC1', 'PC2']].to_numpy())[:]
    y = cur_df['cell_type'].astype('category').cat.codes.to_numpy().reshape(-1,1).astype(np.float32)

    metrics.append([knn_error(X,y), kmeans_ari(X,y), kmeans_nmi(X,y), calc_asw(X,y)])


met_membrane = np.array(metrics)

fig, ax = plt.subplots(2,2, figsize=(10,10))

ax[0][0].plot(pca_df_selected_annotated['t_hours'].unique(), met_membrane[:,0])
ax[0][0].set_title('KNN Error')

ax[0][1].plot(pca_df_selected_annotated['t_hours'].unique(), met_membrane[:,3])
ax[0][1].set_title('ASW')

ax[1][0].plot(pca_df_selected_annotated['t_hours'].unique(), met_membrane[:,1])
ax[1][0].set_title('KMeans ARI')

ax[1][1].plot(pca_df_selected_annotated['t_hours'].unique(), met_membrane[:,2])
ax[1][1].set_title('KMeans NMI')

#ax[0][0].set_ylim(0,1)
#ax[0][1].set_ylim(-1,1)
#ax[1][0].set_ylim(0,0.2)
#ax[1][1].set_ylim(0,0.2)


#plt.savefig('membrane_pca_sep_metrics.pdf')

plt.show()

In [ ]:
# Collect mean and std KNN accuracy for each timepoint using cross-validation
mean_accs_mem = []
std_accs_mem = []

for t in tqdm(pca_df_selected_annotated['t_hours'].unique()):
    # Select data for current timepoint
    mask = (pca_df_selected_annotated['t_hours'] == t)
    cur_df = pca_df_selected_annotated[mask]
    if len(cur_df) < 2:  # skip if not enough samples
        mean_accs_mem.append(np.nan)
        std_accs_mem.append(np.nan)
        continue

    X = StandardScaler().fit_transform(cur_df[['PC1', 'PC2']].to_numpy())
    y = cur_df['cell_type'].astype('category').cat.codes.to_numpy().reshape(-1, 1).astype(np.float32)

    # Get accuracies from multiple folds
    accs = knn_cv_accuracy(X, y)
    mean_accs_mem.append(np.mean(accs))
    std_accs_mem.append(np.std(accs, ddof=1))

# Convert to numpy arrays for plotting
mean_accs = np.array(mean_accs_mem)
std_accs = np.array(std_accs_mem)

### Figure EV1A (membrane features)

In [ ]:
# Fit and transform PCA
pca = PCA(n_components=2)
mem_shape_pca_result_seconddataset = pca.fit_transform(full_df_normalized_cropped2_seconddataset[[feature for feature in ordered_features if feature in feature_categories['membrane_shape']]])  # Drop categorical column

# Convert PCA results to DataFrame
pca_df_seconddataset = pd.DataFrame(mem_shape_pca_result_seconddataset, columns=['PC1', 'PC2'])

pca_df_seconddataset['t_hours'] = full_df_normalized_cropped_seconddataset['t_hours'].values
pca_df_seconddataset['Spot track ID relabelled'] = full_df_normalized_cropped_seconddataset['Spot track ID relabelled'].values
pca_df_seconddataset['cell_type'] = full_df_normalized_cropped_seconddataset['cell_type'].values

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df_seconddataset['PC1'], pca_df_seconddataset['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df_seconddataset['t_hours'],  # Color by time
    cmap='viridis',
    rasterized = True
)

plt.colorbar(label='Time (hours)')  # Add color legend for time
plt.title('Membrane shape PCA by time')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-10,10))
plt.ylim((-10,8))
#plt.savefig('plots_15_03/mem_shape_PCA_seconddataset.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()


In [ ]:
mem_shape_features = [feature for feature in ordered_features if feature in feature_categories['membrane_shape']]

# Combine the original data and the PCA results
combined_df = pd.concat([full_df_normalized_cropped2.reset_index(drop=True), pca_df[['PC1', 'PC2', 't_hours']]], axis=1)

# Compute correlation matrix
correlation_with_pcs = combined_df.corr(method='spearman').loc[mem_shape_features, ['PC1', 'PC2']]

# Sort by absolute correlation with PC1
correlation_sorted_membrane = correlation_with_pcs.reindex(
    correlation_with_pcs['PC1'].abs().sort_values(ascending=False).index
)

### Figure 3E (membrane features)

In [ ]:
plt.figure(figsize=(8, len(mem_shape_features) * 0.5))
sns.heatmap(
    correlation_sorted_membrane,
    annot=True,
    cmap='coolwarm',
    center=0,
    vmin=-1,     # Minimum value on the color scale
    vmax=1       # Maximum value on the color scale
)
plt.title('Membrane shape')
plt.tight_layout()
#plt.savefig('plots_15_03/mem_shape_PCA_featurecorrelation.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()


### Figure 3D, Figure EV1B and Figure EV2B (nucleus features)

In [ ]:
# Fit and transform PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(full_df_normalized_cropped2[[feature for feature in ordered_features if feature in feature_categories['nucleus_shape']]])  # Drop categorical column

# Convert PCA results to DataFrame
pca_df = pd.DataFrame(pca_result, columns=['PC1', 'PC2'])

pca_df['t_hours'] = full_df_normalized_cropped['t_hours'].values
pca_df['Spot track ID relabelled'] = full_df_normalized_cropped['Spot track ID relabelled'].values
pca_df['cell_type'] = full_df_normalized_cropped['cell_type'].values

pca_df_selected_annotated = pca_df[(pca_df['cell_type'] != 'unknown')]

pca_df_selected_annotated = pca_df_selected_annotated.sort_values(['cell_type', 't_hours']).reset_index(drop=True)

pca_df_selected = pca_df[(pca_df['Spot track ID relabelled'].isin(random_choices))]

pca_df_selected = pca_df_selected.sort_values(['Spot track ID relabelled', 't_hours']).reset_index(drop=True)

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df['PC1'], pca_df['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df['t_hours'],  # Color by time
    cmap='viridis',
    rasterized = True
)

plt.colorbar(label='Time (hours)')  # Add color legend for time
plt.title('Nucleus shape PCA by time')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-8,6))
plt.ylim((-6,6))
#plt.savefig('plots_15_03/nuc_shape_PCA.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

plt.figure(figsize=(4, 4))

for track_id, group in pca_df_selected.groupby('Spot track ID relabelled'):  
    plt.plot(group["PC1"], group["PC2"], label=f'Track {track_id}', linewidth=2, alpha=0.7)

plt.title('Tracks in nucleus shape PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-8,6))
plt.ylim((-6,6))
plt.legend()
#plt.savefig('plots_15_03/nuc_shape_PCA_tracks.pdf')
plt.show()

# Define color mapping for cell types
cell_type_colors = {'basal': 'royalblue', 'goblet': 'orange', 'ic': 'magenta', 'mcc': 'green', 'ssc': 'gold'}

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df_selected_annotated['PC1'], pca_df_selected_annotated['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df_selected_annotated['cell_type'].map(cell_type_colors),  # Color by cell type
    rasterized = True
)

# Create legend handles
legend_patches = [mpatches.Patch(color=color, label=ctype) for ctype, color in cell_type_colors.items()]
plt.legend(handles=legend_patches, title="Cell type", frameon=True)

plt.title('Nucleus shape PCA by cell type')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-8,6))
plt.ylim((-6,6))
#plt.savefig('plots_15_03/nuc_shape_PCA_celltypes.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

### To figure EV2A (plotted later)

In [ ]:
metrics = []

for t in tqdm(pca_df_selected_annotated['t_hours'].unique()):
    cur_df = pca_df_selected_annotated[pca_df_selected_annotated['t_hours'] == t]
    
    X = StandardScaler().fit_transform(cur_df[['PC1', 'PC2']].to_numpy())[:]
    y = cur_df['cell_type'].astype('category').cat.codes.to_numpy().reshape(-1,1).astype(np.float32)

    metrics.append([knn_error(X,y), kmeans_ari(X,y), kmeans_nmi(X,y), calc_asw(X,y)])


met_nucleus = np.array(metrics)

fig, ax = plt.subplots(2,2, figsize=(10,10))

ax[0][0].plot(pca_df_selected_annotated['t_hours'].unique(), met_nucleus[:,0])
ax[0][0].set_title('KNN Error')

ax[0][1].plot(pca_df_selected_annotated['t_hours'].unique(), met_nucleus[:,3])
ax[0][1].set_title('ASW')

ax[1][0].plot(pca_df_selected_annotated['t_hours'].unique(), met_nucleus[:,1])
ax[1][0].set_title('KMeans ARI')

ax[1][1].plot(pca_df_selected_annotated['t_hours'].unique(), met_nucleus[:,2])
ax[1][1].set_title('KMeans NMI')

#ax[0][0].set_ylim(0,1)
#ax[0][1].set_ylim(-1,1)
#ax[1][0].set_ylim(0,0.2)
#ax[1][1].set_ylim(0,0.2)


#plt.savefig('nucleus_pca_sep_metrics.pdf')

plt.show()

In [ ]:
# Collect mean and std KNN accuracy for each timepoint using cross-validation
mean_accs_nuc = []
std_accs_nuc = []

for t in tqdm(pca_df_selected_annotated['t_hours'].unique()):
    # Select data for current timepoint
    mask = (pca_df_selected_annotated['t_hours'] == t)
    cur_df = pca_df_selected_annotated[mask]
    if len(cur_df) < 2:  # skip if not enough samples
        mean_accs_nuc.append(np.nan)
        std_accs_nuc.append(np.nan)
        continue

    X = StandardScaler().fit_transform(cur_df[['PC1', 'PC2']].to_numpy())
    y = cur_df['cell_type'].astype('category').cat.codes.to_numpy().reshape(-1, 1).astype(np.float32)

    # Get accuracies from multiple folds
    accs = knn_cv_accuracy(X, y)
    mean_accs_nuc.append(np.mean(accs))
    std_accs_nuc.append(np.std(accs, ddof=1))

# Convert to numpy arrays for plotting
mean_accs = np.array(mean_accs_nuc)
std_accs = np.array(std_accs_nuc)

### Figure EV1A (nucleus features)

In [ ]:
# Fit and transform PCA
pca = PCA(n_components=2)
pca_result_seconddataset = pca.fit_transform(full_df_normalized_cropped2_seconddataset[[feature for feature in ordered_features if feature in feature_categories['nucleus_shape']]])  # Drop categorical column

# Convert PCA results to DataFrame
pca_df_seconddataset = pd.DataFrame(pca_result_seconddataset, columns=['PC1', 'PC2'])

pca_df_seconddataset['t_hours'] = full_df_normalized_cropped_seconddataset['t_hours'].values
pca_df_seconddataset['Spot track ID relabelled'] = full_df_normalized_cropped_seconddataset['Spot track ID relabelled'].values
pca_df_seconddataset['cell_type'] = full_df_normalized_cropped_seconddataset['cell_type'].values

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df_seconddataset['PC1'], pca_df_seconddataset['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df_seconddataset['t_hours'],  # Color by time
    cmap='viridis',
    rasterized = True
)

plt.colorbar(label='Time (hours)')  # Add color legend for time
plt.title('Nucleus shape PCA by time')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-8,6))
plt.ylim((-6,6))
plt.savefig('plots_15_03/nuc_shape_PCA_seconddataset.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()


### Figure 3E (nucleus features)

In [ ]:
nuc_shape_features = [feature for feature in ordered_features if feature in feature_categories['nucleus_shape']]

# Combine the original data and the PCA results
combined_df = pd.concat([full_df_normalized_cropped2.reset_index(drop=True), pca_df[['PC1', 'PC2', 't_hours']]], axis=1)

# Compute correlation matrix
correlation_with_pcs = combined_df.corr(method='spearman').loc[nuc_shape_features, ['PC1', 'PC2']]

# Sort by absolute correlation with PC1
correlation_sorted_nucleus = correlation_with_pcs.reindex(
    correlation_with_pcs['PC1'].abs().sort_values(ascending=False).index
)

In [ ]:
plt.figure(figsize=(8, len(nuc_shape_features) * 0.5))
sns.heatmap(
    correlation_sorted_nucleus,
    annot=True,
    cmap='coolwarm',
    center=0,
    vmin=-1,     # Minimum value on the color scale
    vmax=1       # Maximum value on the color scale
)
plt.title('Nucleus shape')
plt.tight_layout()
#plt.savefig('plots_15_03/nuc_shape_PCA_featurecorrelation.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()


### Positional features PCA (not used)

In [ ]:
# Fit and transform PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(full_df_normalized_cropped2[[feature for feature in ordered_features if feature in feature_categories['position']]])  # Drop categorical column

# Convert PCA results to DataFrame
pca_df = pd.DataFrame(pca_result, columns=['PC1', 'PC2'])

pca_df['t_hours'] = full_df_normalized_cropped['t_hours'].values
pca_df['Spot track ID relabelled'] = full_df_normalized_cropped['Spot track ID relabelled'].values
pca_df['cell_type'] = full_df_normalized_cropped['cell_type'].values

pca_df_selected_annotated = pca_df[(pca_df['cell_type'] != 'unknown')]

pca_df_selected_annotated = pca_df_selected_annotated.sort_values(['cell_type', 't_hours']).reset_index(drop=True)

pca_df_selected = pca_df[(pca_df['Spot track ID relabelled'].isin(random_choices))]

pca_df_selected = pca_df_selected.sort_values(['Spot track ID relabelled', 't_hours']).reset_index(drop=True)

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df['PC1'], pca_df['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df['t_hours'],  # Color by time
    cmap='viridis',
    rasterized = True
)

plt.colorbar(label='Time (hours)')  # Add color legend for time
plt.title('Positional PCA by time')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-6,6))
plt.ylim((-6,4))
#plt.savefig('plots_15_03/position_PCA.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

plt.figure(figsize=(4, 4))

for track_id, group in pca_df_selected.groupby('Spot track ID relabelled'):  
    plt.plot(group["PC1"], group["PC2"], label=f'Track {track_id}', linewidth=2, alpha=0.7)
    
plt.title('Tracks in positional PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-6,6))
plt.ylim((-6,4))
plt.legend()
#plt.savefig('plots_15_03/position_PCA_tracks.pdf')
plt.show()

# Define color mapping for cell types
cell_type_colors = {'basal': 'royalblue', 'goblet': 'orange', 'ic': 'magenta', 'mcc': 'green', 'ssc': 'gold'}

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df_selected_annotated['PC1'], pca_df_selected_annotated['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df_selected_annotated['cell_type'].map(cell_type_colors),  # Color by cell type
    rasterized = True
)

# Create legend handles
legend_patches = [mpatches.Patch(color=color, label=ctype) for ctype, color in cell_type_colors.items()]
plt.legend(handles=legend_patches, title="Cell type", frameon=True)

plt.title('Positional PCA by cell type')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-6,6))
plt.ylim((-6,4))
#plt.savefig('plots_15_03/position_PCA_celltypes.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
metrics = []

for t in tqdm(pca_df_selected_annotated['t_hours'].unique()):
    cur_df = pca_df_selected_annotated[pca_df_selected_annotated['t_hours'] == t]
    
    X = StandardScaler().fit_transform(cur_df[['PC1', 'PC2']].to_numpy())[:]
    y = cur_df['cell_type'].astype('category').cat.codes.to_numpy().reshape(-1,1).astype(np.float32)

    metrics.append([knn_error(X,y), kmeans_ari(X,y), kmeans_nmi(X,y), calc_asw(X,y)])


met_position = np.array(metrics)

fig, ax = plt.subplots(2,2, figsize=(10,10))

ax[0][0].plot(pca_df_selected_annotated['t_hours'].unique(), met_position[:,0])
ax[0][0].set_title('KNN Error')

ax[0][1].plot(pca_df_selected_annotated['t_hours'].unique(), met_position[:,3])
ax[0][1].set_title('ASW')

ax[1][0].plot(pca_df_selected_annotated['t_hours'].unique(), met_position[:,1])
ax[1][0].set_title('KMeans ARI')

ax[1][1].plot(pca_df_selected_annotated['t_hours'].unique(), met_position[:,2])
ax[1][1].set_title('KMeans NMI')

#ax[0][0].set_ylim(0,1)
#ax[0][1].set_ylim(-1,1)
#ax[1][0].set_ylim(0,0.2)
#ax[1][1].set_ylim(0,0.2)


#plt.savefig('position_pca_sep_metrics.pdf')

plt.show()

In [ ]:
# Collect mean and std KNN accuracy for each timepoint using cross-validation
mean_accs_position = []
std_accs_position = []

for t in tqdm(pca_df_selected_annotated['t_hours'].unique()):
    # Select data for current timepoint
    mask = (pca_df_selected_annotated['t_hours'] == t)
    cur_df = pca_df_selected_annotated[mask]
    if len(cur_df) < 2:  # skip if not enough samples
        mean_accs_position.append(np.nan)
        std_accs_position.append(np.nan)
        continue

    X = StandardScaler().fit_transform(cur_df[['PC1', 'PC2']].to_numpy())
    y = cur_df['cell_type'].astype('category').cat.codes.to_numpy().reshape(-1, 1).astype(np.float32)

    # Get accuracies from multiple folds
    accs = knn_cv_accuracy(X, y)
    mean_accs_position.append(np.mean(accs))
    std_accs_position.append(np.std(accs, ddof=1))

# Convert to numpy arrays for plotting
mean_accs = np.array(mean_accs_position)
std_accs = np.array(std_accs_position)

# Plot mean ± std accuracy over time
plt.figure(figsize=(10, 10))
plt.plot(pca_df_selected_annotated['t_hours'].unique(), mean_accs, color='blue', label='Mean Accuracy')
plt.fill_between(pca_df_selected_annotated['t_hours'].unique(), mean_accs - std_accs, mean_accs + std_accs, color='blue', alpha=0.3, label='±1 std dev')
plt.xlabel('Time (hours)')
#plt.ylabel('KNN Accuracy')
#plt.title('KNN Accuracy Across Timepoints')
#plt.legend()
#plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Fit and transform PCA
pca = PCA(n_components=2)
pca_result_seconddataset = pca.fit_transform(full_df_normalized_cropped2_seconddataset[[feature for feature in ordered_features if feature in feature_categories['position']]])  # Drop categorical column

# Convert PCA results to DataFrame
pca_df_seconddataset = pd.DataFrame(pca_result_seconddataset, columns=['PC1', 'PC2'])

pca_df_seconddataset['t_hours'] = full_df_normalized_cropped_seconddataset['t_hours'].values
pca_df_seconddataset['Spot track ID relabelled'] = full_df_normalized_cropped_seconddataset['Spot track ID relabelled'].values
pca_df_seconddataset['cell_type'] = full_df_normalized_cropped_seconddataset['cell_type'].values


plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df_seconddataset['PC1'], pca_df_seconddataset['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df_seconddataset['t_hours'],  # Color by time
    cmap='viridis',
    rasterized = True
)

plt.colorbar(label='Time (hours)')  # Add color legend for time
plt.title('Positional PCA by time')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-6,6))
plt.ylim((-4,6))
#plt.savefig('plots_15_03/position_PCA_seconddataset.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
pos_shape_features = [feature for feature in ordered_features if feature in feature_categories['position']]

# Combine the original data and the PCA results
combined_df = pd.concat([full_df_normalized_cropped2.reset_index(drop=True), pca_df[['PC1', 'PC2', 't_hours']]], axis=1)

# Compute correlation matrix
correlation_with_pcs = combined_df.corr(method='spearman').loc[pos_shape_features, ['PC1', 'PC2']]

# Sort by absolute correlation with PC1
correlation_sorted_position = correlation_with_pcs.reindex(
    correlation_with_pcs['PC1'].abs().sort_values(ascending=False).index
)

In [ ]:
plt.figure(figsize=(8, len(pos_shape_features) * 0.5))
sns.heatmap(
    correlation_sorted_position,
    annot=True,
    cmap='coolwarm',
    center=0,
    vmin=-1,     # Minimum value on the color scale
    vmax=1       # Maximum value on the color scale
)
plt.title('Position')
plt.tight_layout()
#plt.savefig('plots_15_03/position_PCA_featurecorrelation.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()


### Figure 3D, Figure EV1B and Figure EV2B (movement features)

In [ ]:
# Fit and transform PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(full_df_normalized_cropped2[[feature for feature in ordered_features if feature in feature_categories['movement']]])  # Drop categorical column

# Convert PCA results to DataFrame
pca_df = pd.DataFrame(pca_result, columns=['PC1', 'PC2'])

pca_df['t_hours'] = full_df_normalized_cropped['t_hours'].values
pca_df['Spot track ID relabelled'] = full_df_normalized_cropped['Spot track ID relabelled'].values
pca_df['cell_type'] = full_df_normalized_cropped['cell_type'].values

pca_df_selected_annotated = pca_df[(pca_df['cell_type'] != 'unknown')]

pca_df_selected_annotated = pca_df_selected_annotated.sort_values(['cell_type', 't_hours']).reset_index(drop=True)

pca_df_selected = pca_df[(pca_df['Spot track ID relabelled'].isin(random_choices))]

pca_df_selected = pca_df_selected.sort_values(['Spot track ID relabelled', 't_hours']).reset_index(drop=True)

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df['PC1'], pca_df['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df['t_hours'],  # Color by time
    cmap='viridis',
    rasterized=True
)

plt.colorbar(label='Time (hours)')  # Add color legend for time
plt.title('Motion PCA by time')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-5,15))
plt.ylim((-4,8))
#plt.savefig('plots_15_03/movement_PCA.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

plt.figure(figsize=(4, 4))

for track_id, group in pca_df_selected.groupby('Spot track ID relabelled'):  
    plt.plot(group["PC1"], group["PC2"], label=f'Track {track_id}', linewidth=2, alpha=0.7)
    
plt.title('Tracks in motion PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-5,15))
plt.ylim((-4,8))
plt.legend()
#plt.savefig('plots_15_03/movement_PCA_tracks.pdf')
plt.show()

# Define color mapping for cell types
cell_type_colors = {'basal': 'royalblue', 'goblet': 'orange', 'ic': 'magenta', 'mcc': 'green', 'ssc': 'gold'}

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df_selected_annotated['PC1'], pca_df_selected_annotated['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df_selected_annotated['cell_type'].map(cell_type_colors),  # Color by cell type
    rasterized = True
)

# Create legend handles
legend_patches = [mpatches.Patch(color=color, label=ctype) for ctype, color in cell_type_colors.items()]
plt.legend(handles=legend_patches, title="Cell type", frameon=True)

plt.title('Motion PCA by cell type')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-5,15))
plt.ylim((-4,8))
#plt.savefig('plots_15_03/movement_PCA_celltypes.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

### To figure EV2A (plotted later)

In [ ]:
metrics = []

for t in tqdm(pca_df_selected_annotated['t_hours'].unique()):
    cur_df = pca_df_selected_annotated[pca_df_selected_annotated['t_hours'] == t]
    
    X = StandardScaler().fit_transform(cur_df[['PC1', 'PC2']].to_numpy())[:]
    y = cur_df['cell_type'].astype('category').cat.codes.to_numpy().reshape(-1,1).astype(np.float32)

    metrics.append([knn_error(X,y), kmeans_ari(X,y), kmeans_nmi(X,y), calc_asw(X,y)])


met_movement = np.array(metrics)

fig, ax = plt.subplots(2,2, figsize=(10,10))

ax[0][0].plot(pca_df_selected_annotated['t_hours'].unique(), met_movement[:,0])
ax[0][0].set_title('KNN Error')

ax[0][1].plot(pca_df_selected_annotated['t_hours'].unique(), met_movement[:,3])
ax[0][1].set_title('ASW')

ax[1][0].plot(pca_df_selected_annotated['t_hours'].unique(), met_movement[:,1])
ax[1][0].set_title('KMeans ARI')

ax[1][1].plot(pca_df_selected_annotated['t_hours'].unique(), met_movement[:,2])
ax[1][1].set_title('KMeans NMI')

#ax[0][0].set_ylim(0,1)
#ax[0][1].set_ylim(-1,1)
#ax[1][0].set_ylim(0,0.2)
#ax[1][1].set_ylim(0,0.2)


#plt.savefig('movement_pca_sep_metrics.pdf')

plt.show()

In [ ]:
# Collect mean and std KNN accuracy for each timepoint using cross-validation
mean_accs_mov = []
std_accs_mov = []

for t in tqdm(pca_df_selected_annotated['t_hours'].unique()):
    # Select data for current timepoint
    mask = (pca_df_selected_annotated['t_hours'] == t)
    cur_df = pca_df_selected_annotated[mask]
    if len(cur_df) < 2:  # skip if not enough samples
        mean_accs_mov.append(np.nan)
        std_accs_mov.append(np.nan)
        continue

    X = StandardScaler().fit_transform(cur_df[['PC1', 'PC2']].to_numpy())
    y = cur_df['cell_type'].astype('category').cat.codes.to_numpy().reshape(-1, 1).astype(np.float32)

    # Get accuracies from multiple folds
    accs = knn_cv_accuracy(X, y)
    mean_accs_mov.append(np.mean(accs))
    std_accs_mov.append(np.std(accs, ddof=1))

# Convert to numpy arrays for plotting
mean_accs = np.array(mean_accs_mov)
std_accs = np.array(std_accs_mov)

### Figure EV1A (movement features)

In [ ]:
# Fit and transform PCA
pca = PCA(n_components=2)
pca_result_seconddataset = pca.fit_transform(full_df_normalized_cropped2_seconddataset[[feature for feature in ordered_features if feature in feature_categories['movement']]])  # Drop categorical column

# Convert PCA results to DataFrame
pca_df_seconddataset = pd.DataFrame(pca_result_seconddataset, columns=['PC1', 'PC2'])

pca_df_seconddataset['t_hours'] = full_df_normalized_cropped_seconddataset['t_hours'].values
pca_df_seconddataset['Spot track ID relabelled'] = full_df_normalized_cropped_seconddataset['Spot track ID relabelled'].values
pca_df_seconddataset['cell_type'] = full_df_normalized_cropped_seconddataset['cell_type'].values

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df_seconddataset['PC1'], pca_df_seconddataset['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df_seconddataset['t_hours'],  # Color by time
    cmap='viridis',
    rasterized=True
)

plt.colorbar(label='Time (hours)')  # Add color legend for time
plt.title('Motion PCA by time')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-8,12))
plt.ylim((-4,10))
#plt.savefig('plots_15_03/movement_PCA_seconddataset.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

### Figure 3E (movement features)

In [ ]:
mov_shape_features = [feature for feature in ordered_features if feature in feature_categories['movement']]

# Combine the original data and the PCA results
combined_df = pd.concat([full_df_normalized_cropped2.reset_index(drop=True), pca_df[['PC1', 'PC2', 't_hours']]], axis=1)

# Compute correlation matrix
correlation_with_pcs = combined_df.corr(method='spearman').loc[mov_shape_features, ['PC1', 'PC2']]

# Sort by absolute correlation with PC1
correlation_sorted_motion = correlation_with_pcs.reindex(
    correlation_with_pcs['PC1'].abs().sort_values(ascending=False).index
)

In [ ]:
plt.figure(figsize=(8, len(mov_shape_features) * 0.5))
sns.heatmap(
    correlation_sorted_motion,
    annot=True,
    cmap='coolwarm',
    center=0,
    vmin=-1,     # Minimum value on the color scale
    vmax=1       # Maximum value on the color scale
)
plt.title('Movement')
plt.tight_layout()
#plt.savefig('plots_15_03/movement_PCA_featurecorrelation.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()


### Figure 3D, Figure EV1B and Figure EV2B (all features)

In [ ]:
# Fit and transform PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(full_df_normalized_cropped2[ordered_features])  # Drop categorical column

# Convert PCA results to DataFrame
pca_df = pd.DataFrame(pca_result, columns=['PC1', 'PC2'])

pca_df['t_hours'] = full_df_normalized_cropped['t_hours'].values
pca_df['Spot track ID relabelled'] = full_df_normalized_cropped['Spot track ID relabelled'].values
pca_df['cell_type'] = full_df_normalized_cropped['cell_type'].values

pca_df_selected_annotated = pca_df[(pca_df['cell_type'] != 'unknown')]

pca_df_selected_annotated = pca_df_selected_annotated.sort_values(['cell_type', 't_hours']).reset_index(drop=True)

pca_df_selected = pca_df[(pca_df['Spot track ID relabelled'].isin(random_choices))]

pca_df_selected = pca_df_selected.sort_values(['Spot track ID relabelled', 't_hours']).reset_index(drop=True)

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df['PC1'], pca_df['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df['t_hours'],  # Color by time
    cmap='viridis',
    rasterized=True
)

plt.colorbar(label='Time (hours)')  # Add color legend for time
plt.title('All features PCA by time')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-12,14))
plt.ylim((-6,12))
#plt.savefig('plots_15_03/all_PCA.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

plt.figure(figsize=(4, 4))

for track_id, group in pca_df_selected.groupby('Spot track ID relabelled'):  
    plt.plot(group["PC1"], group["PC2"], label=f'Track {track_id}', linewidth=2, alpha=0.7)
    
plt.title('Tracks in all features PCA')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-12,14))
plt.ylim((-6,12))
plt.legend()
#plt.savefig('plots_15_03/all_PCA_tracks_new.pdf')
plt.show()

# Define color mapping for cell types
cell_type_colors = {'basal': 'royalblue', 'goblet': 'orange', 'ic': 'magenta', 'mcc': 'green', 'ssc': 'gold'}

plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df_selected_annotated['PC1'], pca_df_selected_annotated['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df_selected_annotated['cell_type'].map(cell_type_colors),  # Color by cell type
    rasterized = True
)

# Create legend handles
legend_patches = [mpatches.Patch(color=color, label=ctype) for ctype, color in cell_type_colors.items()]
plt.legend(handles=legend_patches, title="Cell type", frameon=True)

plt.title('All features PCA by cell type')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-12,14))
plt.ylim((-6,12))
#plt.savefig('plots_15_03/all_PCA_celltypes.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

### To figure EV2A (plotted later)

In [ ]:
# Fit PCA with 3 components
pca = PCA(n_components=3)
pca.fit(full_df_normalized_cropped2[ordered_features])

# Explained variance ratio for the first 3 PCs
explained_var = pca.explained_variance_ratio_
total_explained = explained_var.sum()

print(f"Explained variance by first 3 PCs: {explained_var}")
print(f"Total explained variance (first 3 PCs): {total_explained:.3f}")

In [ ]:
metrics = []

for t in tqdm(pca_df_selected_annotated['t_hours'].unique()):
    cur_df = pca_df_selected_annotated[pca_df_selected_annotated['t_hours'] == t]
    
    X = StandardScaler().fit_transform(cur_df[['PC1', 'PC2']].to_numpy())[:]
    y = cur_df['cell_type'].astype('category').cat.codes.to_numpy().reshape(-1,1).astype(np.float32)

    metrics.append([knn_error(X,y), kmeans_ari(X,y), kmeans_nmi(X,y), calc_asw(X,y)])


met_all = np.array(metrics)

fig, ax = plt.subplots(2,2, figsize=(10,10))

ax[0][0].plot(pca_df_selected_annotated['t_hours'].unique(), met_all[:,0])
ax[0][0].set_title('KNN Error')

ax[0][1].plot(pca_df_selected_annotated['t_hours'].unique(), met_all[:,3])
ax[0][1].set_title('ASW')

ax[1][0].plot(pca_df_selected_annotated['t_hours'].unique(), met_all[:,1])
ax[1][0].set_title('KMeans ARI')

ax[1][1].plot(pca_df_selected_annotated['t_hours'].unique(), met_all[:,2])
ax[1][1].set_title('KMeans NMI')

#ax[0][0].set_ylim(0,1)
#ax[0][1].set_ylim(-1,1)
#ax[1][0].set_ylim(0,0.2)
#ax[1][1].set_ylim(0,0.2)


#plt.savefig('allfeatures_pca_sep_metrics.pdf')

plt.show()

In [ ]:
from sklearn.dummy import DummyClassifier

def get_baseline_accuracy(X, y):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    dummy_accs = []
    dummy_balanced_accs = []

    for train_idx, test_idx in skf.split(X, y.ravel()):
        dummy = DummyClassifier(strategy='stratified')  # or 'most_frequent'
        dummy.fit(X[train_idx], y[train_idx])
        y_pred = dummy.predict(X[test_idx])
        dummy_accs.append(accuracy_score(y[test_idx], y_pred))
        dummy_balanced_accs.append(balanced_accuracy_score(y[test_idx], y_pred))
    return dummy_accs, dummy_balanced_accs



In [ ]:
# Collect mean and std KNN accuracy for each timepoint using cross-validation
mean_accs_all = []
std_accs_all = []
accs_all = []
balanced_accs_all = []
dummy_accs_all = []
dummy_balanced_accs_all = []


for t in tqdm(pca_df_selected_annotated['t_hours'].unique()):
    # Select data for current timepoint
    mask = (pca_df_selected_annotated['t_hours'] == t)
    cur_df = pca_df_selected_annotated[mask]
    if len(cur_df) < 2:  # skip if not enough samples
        mean_accs_all.append(np.nan)
        std_accs_all.append(np.nan)
        continue

    X = StandardScaler().fit_transform(cur_df[['PC1', 'PC2']].to_numpy())
    y = cur_df['cell_type'].astype('category').cat.codes.to_numpy().reshape(-1, 1).astype(np.float32)

    # Get accuracies from multiple folds
    accs, balanced_accs = knn_cv_accuracy(X, y)
    dummy_accs, dummy_balaced_accs = get_baseline_accuracy(X, y)

    mean_accs_all.append(np.mean(accs))
    std_accs_all.append(np.std(accs, ddof=1))
    accs_all.append(accs)
    balanced_accs_all.append(balanced_accs)
    dummy_accs_all.append(dummy_accs)
    dummy_balanced_accs_all.append(dummy_balaced_accs)

# Convert to numpy arrays for plotting
mean_accs = np.array(mean_accs_all)
std_accs = np.array(std_accs_all)

In [ ]:
# accs_all is a list of lists (each inner list: fold accuracies for a timepoint)
all_accs_flat = np.concatenate(accs_all)  # flatten to 1D array

mean_acc = np.mean(all_accs_flat)
std_acc = np.std(all_accs_flat, ddof=1)

print(f"Mean accuracy: {mean_acc:.4f}")
print(f"Standard deviation: {std_acc:.4f}")

# accs_all is a list of lists (each inner list: fold accuracies for a timepoint)
all_balanced_accs_flat = np.concatenate(balanced_accs_all)  # flatten to 1D array

mean_balanced_acc = np.mean(all_balanced_accs_flat)
std_balanced_acc = np.std(all_balanced_accs_flat, ddof=1)

print(f"Mean balanced accuracy: {mean_balanced_acc:.4f}")
print(f"Standard balanced deviation: {std_balanced_acc:.4f}")

# accs_all is a list of lists (each inner list: fold accuracies for a timepoint)
all_dummy_accs_flat = np.concatenate(dummy_accs_all)  # flatten to 1D array

mean_dummy_acc = np.mean(all_dummy_accs_flat)
std_dummy_acc = np.std(all_dummy_accs_flat, ddof=1)

print(f"Mean baseline accuracy: {mean_dummy_acc:.4f}")
print(f"Standard deviation baseline: {std_dummy_acc:.4f}")

# accs_all is a list of lists (each inner list: fold accuracies for a timepoint)
all_dummy_balanced_accs_flat = np.concatenate(dummy_balanced_accs_all)  # flatten to 1D array

mean_dummy_balanced_acc = np.mean(all_dummy_balanced_accs_flat)
std_dummy_balanced_acc = np.std(all_dummy_balanced_accs_flat, ddof=1)

print(f"Mean baseline balanced accuracy: {mean_dummy_balanced_acc:.4f}")
print(f"Standard baseline balanced deviation: {std_dummy_balanced_acc:.4f}")

### Figure EV1A (all features)

In [ ]:
# Fit and transform PCA
pca = PCA(n_components=2)
pca_result_seconddataset = pca.fit_transform(full_df_normalized_cropped2_seconddataset[ordered_features])  # Drop categorical column

# Convert PCA results to DataFrame
pca_df_seconddataset = pd.DataFrame(pca_result_seconddataset, columns=['PC1', 'PC2'])

pca_df_seconddataset['t_hours'] = full_df_normalized_cropped_seconddataset['t_hours'].values
pca_df_seconddataset['Spot track ID relabelled'] = full_df_normalized_cropped_seconddataset['Spot track ID relabelled'].values
pca_df_seconddataset['cell_type'] = full_df_normalized_cropped_seconddataset['cell_type'].values


plt.figure(figsize=(4, 4))
scatter = plt.scatter(
    pca_df_seconddataset['PC1'], pca_df_seconddataset['PC2'], 
    s=0.01,  # Smaller point size
    c=pca_df_seconddataset['t_hours'],  # Color by time
    cmap='viridis',
    rasterized=True
)

plt.colorbar(label='Time (hours)')  # Add color legend for time
plt.title('All features PCA by time')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(False)
plt.xlim((-12,14))
plt.ylim((-6,12))
#plt.savefig('plots_15_03/all_PCA_seconddataset.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

### Figure 3E and EV1D (all features)

In [ ]:
# Combine the original data and the PCA results
combined_df = pd.concat([full_df_normalized_cropped2.reset_index(drop=True), pca_df[['PC1', 'PC2', 't_hours']]], axis=1)

# Compute correlation matrix
correlation_with_pcs = combined_df.corr().loc[ordered_features, ['PC1', 'PC2']]

# Sort by absolute correlation with PC1
correlation_sorted_all = correlation_with_pcs.reindex(
    correlation_with_pcs['PC1'].abs().sort_values(ascending=False).index
)

In [ ]:
plt.figure(figsize=(8, len(columns_to_normalize) * 0.5))
sns.heatmap(correlation_sorted_all, annot=True, cmap='coolwarm', center=0)
plt.title('All features')
plt.tight_layout()
#plt.savefig('plots_15_03/allfeatures_PCA_featurecorrelation.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.gridspec as gridspec

# Correlation matrices and titles
correlation_matrices = [
    correlation_sorted_membrane,
    correlation_sorted_nucleus,
    correlation_sorted_position,
    correlation_sorted_motion
]

titles = ['Membrane shape', 'Nucleus shape', 'Position', 'Movement']
row_counts = [len(df) for df in correlation_matrices]
max_rows = max(row_counts)

# Create GridSpec with 1 row, 4 columns
fig = plt.figure(figsize=(4 * 4, max_rows * 0.6))  # width scales with #plots, height with max rows
gs = gridspec.GridSpec(nrows=1, ncols=4, figure=fig, wspace=0.4)

# Loop and plot each heatmap
axes = []
for i in range(4):
    ax = fig.add_subplot(gs[0, i])
    sns.heatmap(
        correlation_matrices[i],
        annot=True,
        cmap='coolwarm',
        center=0,
        vmin=-1,
        vmax=1,
        cbar=False,
        ax=ax
    )
    ax.set_title(titles[i])
    ax.tick_params(axis='y', labelrotation=0)
    axes.append(ax)

# Shared colorbar to the right
cbar_ax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
sns.heatmap(
    correlation_matrices[0],  # dummy just for colorbar
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    cbar=True,
    cbar_ax=cbar_ax,
    xticklabels=False,
    yticklabels=False
)

plt.tight_layout(rect=[0, 0, 0.9, 1])
#plt.savefig("plots_15_03/horizontal_feature_PCA_correlations.pdf", dpi=300, bbox_inches="tight")
plt.show()


### Figure EV2A

In [ ]:
# Define all metrics
all_metrics_dict = {
    'membrane_shape': met_membrane,
    'nucleus_shape': met_nucleus,
    'position': met_position,
    'movement': met_movement,
    'all': met_all
}

# Category colors (add 'all' manually)
category_colors = {
    'membrane_shape': 'royalblue', 
    'nucleus_shape': 'orange',
    'position': 'green',
    'movement': 'magenta',
    'all': 'gray'
}

metric_names = ['KNN Error', 'KMeans ARI', 'KMeans NMI', 'ASW']
n_metrics = len(metric_names)
timepoints = pca_df_selected_annotated['t_hours'].unique()

fig, axs = plt.subplots(2, 2, figsize=(12, 10))
axs = axs.flatten()

for i in range(n_metrics):
    ax = axs[i]
    for group, metric_array in all_metrics_dict.items():
        ax.plot(
            timepoints,
            metric_array[:, i],
            label=group.replace('_', ' ').title(),
            color=category_colors[group],
            alpha=0.7
        )
    ax.set_title(metric_names[i])
    ax.set_xlabel("Time (hours)")
    ax.set_ylabel(metric_names[i])
    ax.grid(True)

# Finalize
fig.suptitle("Separability Metrics Over Time by Feature Category", fontsize=14)
fig.legend(title="Feature Group", bbox_to_anchor=(1.05, 0.5), loc="center left")
fig.tight_layout(rect=[0, 0, 0.85, 1])  # make space for legend
#plt.savefig("plots_15_03/separability_metrics_over_time.pdf", dpi=300, bbox_inches="tight") 
plt.show()


In [ ]:
# Suppose you have accuracy arrays for each feature category, e.g.:
# accs_mem, accs_nuc, accs_position, accs_mov, accs_all
# Each is a numpy array of shape (n_timepoints, n_folds)

# Example: accs_mem = np.array([[fold1_acc, fold2_acc, ...], ...])  # shape: (n_timepoints, n_folds)

mean_category_accs = {
    'Membrane': mean_accs_mem,
    'Nucleus': mean_accs_nuc,
    'Position': mean_accs_position,
    'Movement': mean_accs_mov,
    'All': mean_accs_all
}

std_category_accs = {
    'Membrane': std_accs_mem,
    'Nucleus': std_accs_nuc,
    'Position': std_accs_position,
    'Movement': std_accs_mov,
    'All': std_accs_all
}

category_colors = {
    'Membrane': 'royalblue',
    'Nucleus': 'orange',
    'Position': 'green',
    'Movement': 'magenta',
    'All': 'gray'
}

plt.figure(figsize=(5, 5))

timepoints = pca_df_selected_annotated['t_hours'].unique()

for cat, accs in mean_category_accs.items():
    # Convert to numpy arrays for plotting
    mean_accs = np.array(accs)
    std_accs = np.array(std_category_accs[cat])
    plt.plot(timepoints, mean_accs, label=cat, color=category_colors[cat])
    plt.fill_between(timepoints, mean_accs - std_accs, mean_accs + std_accs, color=category_colors[cat], alpha=0.2)

#plt.xlabel('Time (hours)')
#plt.ylabel('KNN Accuracy')
plt.title('KNN Accuracy')
#plt.legend()
plt.tight_layout()
#plt.savefig("plots_15_03/knn_accuracy_over_time.pdf", dpi=300, bbox_inches="tight") 
plt.show()

In [ ]:
# To get accs_mem, accs_nuc, accs_position, accs_mov, accs_all:
# For each feature category, collect the raw fold accuracies for each timepoint.

def get_knn_fold_accuracies(pca_df_selected_annotated, time_col='t_hours'):
    accs_per_time = []
    for t in tqdm(pca_df_selected_annotated[time_col].unique()):
        mask = (pca_df_selected_annotated[time_col] == t)
        cur_df = pca_df_selected_annotated[mask]
        if len(cur_df) < 2:
            accs_per_time.append([np.nan]*5)  # 5 folds, all nan
            continue
        X = StandardScaler().fit_transform(cur_df[['PC1', 'PC2']].to_numpy())
        y = cur_df['cell_type'].astype('category').cat.codes.to_numpy().reshape(-1, 1).astype(np.float32)
        accs = knn_cv_accuracy(X, y)  # returns list of fold accuracies
        accs_per_time.append(accs)
    return np.array(accs_per_time)

# Example usage:
# accs_mem = get_knn_fold_accuracies(pca_df_selected_annotated_membrane)
# accs_nuc = get_knn_fold_accuracies(pca_df_selected_annotated_nucleus)
# accs_position = get_knn_fold_accuracies(pca_df_selected_annotated_position)
# accs_mov = get_knn_fold_accuracies(pca_df_selected_annotated_movement)
# accs_all = get_knn_fold_accuracies(pca_df_selected_annotated_all)

In [ ]:
# met_all columns: [KNN Error, KMeans ARI, KMeans NMI, ASW]
mean_metrics = np.mean(met_all, axis=0)
sem_metrics = np.std(met_all, axis=0, ddof=1) / np.sqrt(met_all.shape[0])

# KNN accuracy range
knn_accuracies = 1 - met_all[:, 0]
knn_acc_min = knn_accuracies.min()
knn_acc_max = knn_accuracies.max()

print(f"Mean ARI = {mean_metrics[1]:.3f} ± {sem_metrics[1]:.3f}")
print(f"Mean NMI = {mean_metrics[2]:.3f} ± {sem_metrics[2]:.3f}")
print(f"Mean ASW = {mean_metrics[3]:.3f} ± {sem_metrics[3]:.3f}")
print(f"KNN accuracy ranged from {knn_acc_min*100:.1f}% to {knn_acc_max*100:.1f}% across folds")
print(f"Mean KNN accuracy = {knn_accuracies.mean()*100:.1f}% ± {knn_accuracies.std(ddof=1)/np.sqrt(len(knn_accuracies))*100:.1f}%")

In [ ]:
# met_all columns: [KNN Error, KMeans ARI, KMeans NMI, ASW]
mean_metrics = np.mean(met_position, axis=0)
sem_metrics = np.std(met_position, axis=0, ddof=1) / np.sqrt(met_position.shape[0])

# KNN accuracy range
knn_accuracies = 1 - met_position[:, 0]
knn_acc_min = knn_accuracies.min()
knn_acc_max = knn_accuracies.max()

print(f"Mean ARI = {mean_metrics[1]:.3f} ± {sem_metrics[1]:.3f}")
print(f"Mean NMI = {mean_metrics[2]:.3f} ± {sem_metrics[2]:.3f}")
print(f"Mean ASW = {mean_metrics[3]:.3f} ± {sem_metrics[3]:.3f}")
print(f"KNN accuracy ranged from {knn_acc_min*100:.1f}% to {knn_acc_max*100:.1f}% across folds")
print(f"Mean KNN accuracy = {knn_accuracies.mean()*100:.1f}% ± {knn_accuracies.std(ddof=1)/np.sqrt(len(knn_accuracies))*100:.1f}%")

### Figure 3G

In [ ]:
from scipy.stats import spearmanr

feature_corrs = {feature: [] for feature in ordered_features}

for cell in tqdm(full_df_normalized_cropped['Track ID_y'].unique()):
    cell_df = full_df_normalized_cropped[full_df_normalized_cropped['Track ID_y'] == cell]
    if (cell_df['t_hours'].max() - cell_df['t_hours'].min()) >= 1:

        for feature in ordered_features:
            if feature in cell_df.columns:
                try:
                    corr, p = spearmanr(cell_df[feature], cell_df['t_hours'])
                    if not np.isnan(corr):
                        feature_corrs[feature].append(corr)
                except Exception as e:
                    pass  # skip feature if correlation fails

In [ ]:
# Long-form DataFrame for seaborn boxplot
plot_df = pd.DataFrame([
    {'Feature': feature, 'Correlation': corr}
    for feature, corrs in feature_corrs.items() if feature not in []
    for corr in corrs
])

In [ ]:
# Map each feature to its category
feature_to_category = {
    feature: category
    for category, features in feature_categories.items()
    for feature in features
}

# Add 'Category' column to the plot_df
plot_df['Category'] = plot_df['Feature'].map(feature_to_category)

In [ ]:
from scipy.stats import ttest_1samp, wilcoxon

significance_results = {}

for feature, corrs in feature_corrs.items():
    # Remove NaNs
    corrs = np.array(corrs)
    corrs = corrs[~np.isnan(corrs)]
    if len(corrs) > 0:
        # t-test
        t_stat, p_ttest = ttest_1samp(corrs, 0)
        # Wilcoxon signed-rank test (non-parametric)
        try:
            w_stat, p_wilcoxon = wilcoxon(corrs)
        except ValueError:
            p_wilcoxon = np.nan  # Wilcoxon fails if all values are identical
        significance_results[feature] = {
            't_test_p': p_ttest,
            'wilcoxon_p': p_wilcoxon,
            'mean_corr': np.mean(corrs),
            'median_corr': np.median(corrs),
            'n_tracks': len(corrs)
        }

# Example: print features with significant correlation (p < 0.05)
for feature, res in significance_results.items():
    if res['wilcoxon_p'] < 0.05:
        print(f"{feature}: median={res['median_corr']:.2f}, p={res['wilcoxon_p']:.3g}")

In [ ]:
# Add significance stars to printout and prepare for boxplot annotation
def significance_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''

# Print with stars
for feature, res in significance_results.items():
    stars = significance_stars(res['wilcoxon_p'])
    if stars:
        print(f"{feature}: median={res['median_corr']:.2f}, p={res['wilcoxon_p']:.3g} {stars}")

# For boxplot annotation:
# Create a dict mapping feature to stars
feature_significance = {feature: significance_stars(res['wilcoxon_p']) for feature, res in significance_results.items()}

# Example: annotate boxplot with stars above each box
for i, feature in enumerate(ordered_features):
    star = feature_significance.get(feature, '')
    if star:
        # Find y-position for annotation (e.g., max value for that feature)
        y_pos = plot_df[plot_df['Feature'] == feature]['Correlation'].max()
        ax.text(i, y_pos + 0.05, star, ha='center', va='bottom', color='black', fontsize=14)

In [ ]:
plt.figure(figsize=(12, 6))

# Desired legend order based on your dict keys
category_order = list(feature_categories.keys())

ax = sns.boxplot(
    data=plot_df,
    x='Feature',
    y='Correlation',
    hue='Category',         # Color by category
    order=ordered_features, # Maintain order from your dict
    hue_order=category_order,
    palette = {
    'membrane_shape': 'royalblue',
    'nucleus_shape': 'orange',
    'position': 'green',
    'movement': 'magenta'
    },
    showfliers=False,
    width=0.6   # <-- Try 0.6 to 1.0 for wider boxes
)

# Set box transparency
for patch in ax.patches:
    patch.set_alpha(0.8)  # adjust as needed

# Loop through features to add significance asterisks
for i, feature in enumerate(ordered_features):
    star = feature_significance.get(feature, '')
    if star:
        # Get max y-value for that feature (for all categories)
        y_vals = plot_df[plot_df['Feature'] == feature]['Correlation']
        if len(y_vals) == 0:
            continue
        y_pos = y_vals.max()

        # Optional: Add a small buffer to prevent overlap with box
        #y_buffer = 0.00005 * (plot_df['Correlation'].max() - plot_df['Correlation'].min())
        ax.text(
            i, y_pos - 0.02, star,
            ha='center', va='bottom',
            color='black', fontsize=8
        )

plt.xticks(rotation=90, fontsize=8)
plt.title('Time-feature Spearman rank correlation')
plt.legend(
    title='Feature Category',
    bbox_to_anchor=(0.5, -0.5),  # centered above the plot
    loc='upper center',
    ncol=len(feature_categories),  # one column per category
    frameon=False
)
plt.tight_layout()
plt.xlabel("")
plt.yticks(np.arange(-1, 1.05, 0.5))
#plt.savefig('time_feature_correlation_withsignificances.pdf', dpi=300, format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
from scipy.stats import ttest_1samp, wilcoxon
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd

results = []

for feature, corrs in feature_corrs.items():
    corrs = np.array(corrs)
    corrs = corrs[~np.isnan(corrs)]
    if len(corrs) > 0:
        # Fisher Z-transform
        z_corrs = np.arctanh(np.clip(corrs, -0.999999, 0.999999))
        # One-sample t-test on Z-scores
        t_stat, p_ttest = ttest_1samp(z_corrs, 0)
        # Wilcoxon signed-rank test on raw rhos
        try:
            w_stat, p_wilcoxon = wilcoxon(corrs)
        except ValueError:
            p_wilcoxon = np.nan
        results.append({
            'feature': feature,
            'mean_rho': np.mean(corrs),
            'uncorr_p_ttest': p_ttest,
            'uncorr_p_wilcoxon': p_wilcoxon
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)

# Multiple testing correction (Benjamini-Hochberg FDR)
results_df['fdr_p_ttest'] = multipletests(results_df['uncorr_p_ttest'], method='fdr_bh')[1]
results_df['fdr_p_wilcoxon'] = multipletests(results_df['uncorr_p_wilcoxon'], method='fdr_bh')[1]

results_df['significant_ttest'] = results_df['fdr_p_ttest'] < 0.05
results_df['significant_wilcoxon'] = results_df['fdr_p_wilcoxon'] < 0.05

results_df = results_df[['feature', 'mean_rho', 'uncorr_p_ttest', 'uncorr_p_wilcoxon', 'fdr_p_ttest', 'fdr_p_wilcoxon', 'significant_ttest', 'significant_wilcoxon']]
results_df.sort_values('fdr_p_ttest', inplace=True)
results_df.reset_index(drop=True, inplace=True)

results_df

In [ ]:
from scipy.stats import ttest_1samp, wilcoxon
import statsmodels.stats.multitest as smm

results = []
for feature, corrs in feature_corrs.items():
    
    # Fisher Z-transform
    zs = np.arctanh(corrs)
    
    # One-sample t-test on Z-scores
    t_stat, p_val_t = ttest_1samp(zs, 0)
    
    # Wilcoxon signed-rank test on raw correlations
    # Only use if you have at least 5 non-zero differences
    try:
        w_stat, p_val_w = wilcoxon(corrs)
    except ValueError:
        w_stat, p_val_w = np.nan, np.nan
    
    results.append({
        'feature': feature,
        'n_tracks': len(corrs),
        'mean_rho': np.mean(corrs),
        'uncorr_p_ttest': p_val_t,
        'uncorr_p_wilcoxon': p_val_w
    })

res_df = pd.DataFrame(results)

# Multiple-testing correction (False Discovery Rate)
res_df['fdr_p_ttest'] = smm.multipletests(res_df['uncorr_p_ttest'], method='fdr_bh')[1]
res_df['fdr_p_wilcoxon'] = smm.multipletests(res_df['uncorr_p_wilcoxon'].fillna(1), method='fdr_bh')[1]
res_df['significant_ttest'] = res_df['fdr_p_ttest'] < 0.05
res_df['significant_wilcoxon'] = res_df['fdr_p_wilcoxon'] < 0.05

# Display the results
#import ace_tools as tools; tools.display_dataframe_to_user("Feature Correlation Significance", res_df)


### Figure 4F

In [ ]:
from matplotlib.collections import LineCollection

In [ ]:
top_feature_per_category = {
    'membrane_shape': 'mem_Surface_Area',
    'nucleus_shape': 'nuc_Eccentricity_Comp_First',
    'movement': 'Speed',
    'position': 'POSITION_Z_norm'
}

In [ ]:
# Function to format feature names
def format_feature_name(feature):
    formatted = feature.replace('_', ' ')  # Replace underscores with spaces
    formatted = formatted.replace('mem', 'membrane').replace('nuc', 'nucleus')  # Replace abbreviations
    formatted = formatted.capitalize()  # Capitalize first letter, rest lowercase
    return formatted

# Define color mapping for cell types
cell_type_colors = {'basal': 'royalblue', 'goblet': 'orange', 'ic': 'magenta', 'mcc': 'green', 'ssc': 'gold'}

# Define manual y-axis limits per row
y_limits = {0: (-4, 3), 1: (-3, 3), 2: (-3, 8), 3: (-3, 2)}

# Create subplots (4 rows for categories, 5 columns for cell types)
fig, axes = plt.subplots(nrows=4, ncols=5, figsize=(20, 15), sharex=True, sharey=False)

# Flatten axes for easier indexing
axes = axes.flatten()

# Loop through each cell type (columns)
for col_idx, (cell_type, color) in enumerate(cell_type_colors.items()):
    
    # Subset dataframe for the current cell type
    df_cell = selected_df[selected_df['cell_type'] == cell_type]
    
    # Loop through each feature category (rows)
    for row_idx, (category, feature) in enumerate(top_feature_per_category.items()):
        
        # Get the corresponding subplot
        ax = axes[row_idx * 5 + col_idx]  # 4 rows * 5 columns
        
        # Plot the highest PCA-loading feature for this category
        sns.lineplot(data=df_cell, x='t_hours', y=feature, ax=ax, color=color)

        # Set consistent y-limits for the row
        ax.set_ylim(y_limits[row_idx])

        # Increase font sizes
        ax.set_title(f"{cell_type}" if row_idx == 0 else "", fontsize=16)  # Larger title font
        ax.set_xlabel("Time (h)", fontsize=16)  # Larger x-label
        ax.set_ylabel(format_feature_name(feature) if col_idx == 0 else "", fontsize=16)  # Larger y-label

        # Increase tick label font size
        ax.tick_params(axis='both', labelsize=16)  

# Adjust layout and show the figure
plt.tight_layout()
#plt.savefig("D:/for_figures/single_cell_features.pdf", bbox_inches='tight')
plt.show()